First, we need to install the Kaggle API library.

In [1]:
%pip install kaggle

Now, we can download the dataset using the competition name.

In [9]:
import os

# Set up Kaggle API credentials by setting the KAGGLE_CONFIG_DIR environment variable
# Make sure to upload your kaggle.json file to the directory specified below
os.environ['KAGGLE_CONFIG_DIR'] = '/content/'

# Download the dataset
!kaggle competitions download -c playground-series-s5e10

  0% 0.00/7.76M [00:00<?, ?B/s]
100% 7.76M/7.76M [00:00<00:00, 976MB/s]


Next, we'll unzip the downloaded files.

In [10]:
import zipfile

# Unzip the downloaded files
zip_files = [f for f in os.listdir() if f.endswith('.zip')]
for zip_file in zip_files:
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall()
    os.remove(zip_file) # Remove the zip file after extraction

Finally, we'll load the data into pandas DataFrames.

In [11]:
import pandas as pd

# Load the training and test data
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

    print("Training data head:")
    display(train_df.head())

    print("\nTest data head:")
    display(test_df.head())

except FileNotFoundError:
    print("Make sure the train.csv and test.csv files were extracted correctly.")

Training data head:


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56



Test data head:


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3


In [12]:
# Drop the 'id' column from both dataframes
train_df = train_df.drop('id', axis=1)
test_df = test_df.drop('id', axis=1)

print("Training data head after dropping 'id':")
display(train_df.head())

print("\nTest data head after dropping 'id':")
display(test_df.head())

Training data head after dropping 'id':


,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56



Test data head after dropping 'id':


,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3


first we try to predict considering it as a classification problem, we convert probabilities into 0 and 1 with threshold aa 0.5

# Task
Load the data from "https://www.kaggle.com/competitions/playground-series-s5e10", drop the 'id' column from both train and test datasets, build a simple Decision Tree model, perform 5-fold cross-validation and GridSearchCV to find the best parameters, and evaluate the model.

## Prepare data

### Subtask:
Handle categorical features by encoding them.


**Reasoning**:
Identify categorical columns and apply one-hot encoding to both train and test dataframes, then split the features and target variable in the training data.



In [13]:
# Identify categorical columns
categorical_cols = train_df.select_dtypes(include=['object', 'bool']).columns
print("Categorical columns:", categorical_cols)

# Apply one-hot encoding using pandas get_dummies
train_df_encoded = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Ensure columns match between train and test after encoding
# Add missing columns to the test set and fill with 0
missing_cols = set(train_df_encoded.columns) - set(test_df_encoded.columns)
for c in missing_cols:
    test_df_encoded[c] = 0
# Ensure the order of columns is the same
test_df_encoded = test_df_encoded[train_df_encoded.columns]

# Separate features (X) and target variable (y)
X_train = train_df_encoded.drop('accident_risk', axis=1)
y_train = train_df_encoded['accident_risk']
X_test = test_df_encoded.drop('accident_risk', axis=1) # Drop the target from test as well

print("\nEncoded training data head:")
display(X_train.head())

print("\nEncoded test data head:")
display(X_test.head())

print("\nTraining target variable head:")
display(y_train.head())

Categorical columns: Index(['road_type', 'lighting', 'weather', 'road_signs_present', 'public_road',
       'time_of_day', 'holiday', 'school_season'],
      dtype='object')

Encoded training data head:


,num_lanes,curvature,speed_limit,num_reported_accidents,road_type_rural,road_type_urban,lighting_dim,lighting_night,weather_foggy,weather_rainy,road_signs_present_True,public_road_True,time_of_day_evening,time_of_day_morning,holiday_True,school_season_True
0,2,0.06,35,1,False,True,False,False,False,True,False,True,False,False,False,True
1,4,0.99,35,0,False,True,False,False,False,False,True,False,True,False,True,True
2,4,0.63,70,2,True,False,True,False,False,False,False,True,False,True,True,False
3,4,0.07,35,1,False,False,True,False,False,True,True,True,False,True,False,False
4,1,0.58,60,1,True,False,False,False,True,False,False,False,True,False,True,False



Encoded test data head:


,num_lanes,curvature,speed_limit,num_reported_accidents,road_type_rural,road_type_urban,lighting_dim,lighting_night,weather_foggy,weather_rainy,road_signs_present_True,public_road_True,time_of_day_evening,time_of_day_morning,holiday_True,school_season_True
0,2,0.34,45,1,False,False,False,True,False,False,True,True,False,False,True,True
1,3,0.04,45,0,False,True,True,False,True,False,True,False,False,False,True,False
2,2,0.59,35,1,False,True,True,False,False,False,True,False,False,False,True,True
3,4,0.95,35,2,True,False,False,False,False,True,False,False,False,False,False,False
4,2,0.86,35,3,False,False,False,False,False,False,True,False,True,False,False,True



Training target variable head:


,accident_risk
0,0.13
1,0.35
2,0.30
3,0.21
4,0.56


## Agent

### Subtask:
Define the Decision Tree model and the parameter grid for GridSearchCV.


**Reasoning**:
Define the Decision Tree model and the parameter grid for GridSearchCV.



In [14]:
from sklearn.tree import DecisionTreeClassifier

# Instantiate a DecisionTreeClassifier
dt_model = DecisionTreeClassifier(random_state=42)

# Define the parameter grid
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Decision Tree model instantiated.")
print("Parameter grid defined:")
print(param_grid)

Decision Tree model instantiated.
Parameter grid defined:
{'max_depth': [3, 5, 7, 10], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}


## Perform gridsearchcv

### Subtask:
Use GridSearchCV with 5-fold cross-validation to find the best parameters.


**Reasoning**:
Use GridSearchCV with 5-fold cross-validation to find the best parameters.



In [15]:
from sklearn.model_selection import GridSearchCV

# Instantiate GridSearchCV
grid_search = GridSearchCV(estimator=dt_model, param_grid=param_grid, cv=5, scoring='accuracy') # Using accuracy as a placeholder, will change later if needed.

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train > 0.5) # Convert accident_risk to binary for classification

# Print the best parameters
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)

Best parameters found by GridSearchCV:
{'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10}


## Evaluate the model

### Subtask:
Evaluate the performance of the best model on the training data using cross-validation.


**Reasoning**:
Evaluate the performance of the best model on the training data using cross-validation.



In [16]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

# Instantiate a DecisionTreeClassifier with the best parameters found by GridSearchCV
best_dt_model = DecisionTreeClassifier(**grid_search.best_params_, random_state=42)

# Use cross_val_score to perform 5-fold cross-validation on the training data
cv_scores = cross_val_score(best_dt_model, X_train, y_train > 0.5, cv=5, scoring='accuracy')

# Print the cross-validation scores and the mean cross-validation score
print("Cross-validation scores:", cv_scores)
print("Mean cross-validation score:", cv_scores.mean())

Cross-validation scores: [0.93810779 0.93831059 0.93835888 0.93944047 0.93921777]
Mean cross-validation score: 0.938687099531186


## Summary:

### Data Analysis Key Findings

*   Categorical features were successfully identified and one-hot encoded in both the training and testing datasets.
*   A Decision Tree classifier was instantiated and a parameter grid for `max_depth`, `min_samples_split`, and `min_samples_leaf` was defined for hyperparameter tuning.
*   GridSearchCV with 5-fold cross-validation was used to find the best parameters for the Decision Tree model. The best parameters found were `max_depth`: 10, `min_samples_leaf`: 4, and `min_samples_split`: 10.
*   The best Decision Tree model achieved a mean cross-validation accuracy of approximately 0.9387 on the training data using 5-fold cross-validation.

### Insights or Next Steps

*   The model demonstrates good performance on the training data based on cross-validation. The next step is to evaluate the model's performance on the unseen test data.
*   Consider exploring other classification metrics beyond accuracy, such as precision, recall, and F1-score, which might be more informative depending on the specific business problem and the potential class imbalance in the data.


lets compare the probabilities of the predicition with accident_risk

In [17]:
# Train the best model on the entire training data
best_dt_model.fit(X_train, y_train > 0.5)

# Get predicted probabilities for the training data
# Since this is a classification model, we get probabilities for each class
# We are interested in the probability of the positive class (accident_risk > 0.5)
train_pred_proba = best_dt_model.predict_proba(X_train)[:, 1]

# Now you can compare train_pred_proba with y_train
print("Predicted probabilities for the first 10 training samples:")
display(train_pred_proba[:10])

print("\nActual accident_risk for the first 10 training samples:")
display(y_train[:10])

# You can also create a DataFrame to easily compare them side-by-side
comparison_df = pd.DataFrame({'Actual Accident Risk': y_train, 'Predicted Probability (>0.5)': train_pred_proba})
print("\nComparison of actual accident risk and predicted probability:")
display(comparison_df.head())

Predicted probabilities for the first 10 training samples:


array([0.00000000e+00, 0.00000000e+00, 2.36808237e-02, 0.00000000e+00,
       2.71166134e-01, 1.00000000e+00, 4.62463388e-04, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00])


Actual accident_risk for the first 10 training samples:


,accident_risk
0,0.13
1,0.35
2,0.30
3,0.21
4,0.56
5,0.61
6,0.20
7,0.16
8,0.18
9,0.15



Comparison of actual accident risk and predicted probability:


,Actual Accident Risk,Predicted Probability (>0.5)
0,0.13,0.000000
1,0.35,0.000000
2,0.30,0.023681
3,0.21,0.000000
4,0.56,0.271166


# Task
Perform regression analysis to predict 'accident_risk' using the provided dataset.

## Prepare data

### Subtask:
Ensure the target variable is suitable for regression.


**Reasoning**:
Examine the data type and display descriptive statistics of the 'accident_risk' column to ensure it is suitable for regression.



In [18]:
# Examine the data type of the 'accident_risk' column
print("Data type of 'accident_risk' column:", y_train.dtype)

# Display descriptive statistics for the 'accident_risk' column
print("\nDescriptive statistics for 'accident_risk' column:")
display(y_train.describe())

Data type of 'accident_risk' column: float64

Descriptive statistics for 'accident_risk' column:


,accident_risk
count,517754.000000
mean,0.352377
std,0.166417
min,0.000000
25%,0.230000
50%,0.340000
75%,0.460000
max,1.000000


## Define regression model and parameters

### Subtask:
Choose a regression model and define a parameter grid for hyperparameter tuning.


**Reasoning**:
Instantiate a Linear Regression model.



In [19]:
from sklearn.linear_model import LinearRegression

# Instantiate a Linear Regression model
lr_model = LinearRegression()

print("Linear Regression model instantiated.")

Linear Regression model instantiated.


## Train and evaluate the model

### Subtask:
Train the Linear Regression model on the entire training data and evaluate its performance using appropriate regression metrics.


**Reasoning**:
Train the Linear Regression model, make predictions on the training data, and evaluate the model using MAE, MSE, and R2 score.



In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train the Linear Regression model
lr_model.fit(X_train, y_train)

# Make predictions on the training data
train_predictions = lr_model.predict(X_train)

# Calculate evaluation metrics
mae = mean_absolute_error(y_train, train_predictions)
mse = mean_squared_error(y_train, train_predictions)
r2 = r2_score(y_train, train_predictions)

# Print the evaluation metrics
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared Score (R2): {r2}")

Mean Absolute Error (MAE): 0.058352552638198676
Mean Squared Error (MSE): 0.005401350256156147
R-squared Score (R2): 0.8049674422737241


## Make predictions on the test data

### Subtask:
Use the trained Linear Regression model to make predictions on the unseen test data.


**Reasoning**:
Use the trained Linear Regression model to make predictions on the unseen test data and display the first few predictions.



In [22]:
# Use the trained lr_model to make predictions on the test data
test_predictions = lr_model.predict(X_test)

# Display the first few predicted values rounded to 2 decimal places
print("First 10 predicted accident risk values for the test data (rounded to 2 decimal places):")
display(test_predictions[:10].round(2))

# To compare with actual values, we'll show the comparison for the training data
train_predictions_rounded = lr_model.predict(X_train).round(2)
comparison_df_rounded = pd.DataFrame({'Actual Accident Risk': y_train.round(2), 'Predicted Accident Risk': train_predictions_rounded})

print("\nComparison of actual and predicted accident risk on training data (rounded to 2 decimal places):")
display(comparison_df_rounded.head(10))

First 10 predicted accident risk values for the test data (rounded to 2 decimal places):


array([0.37, 0.16, 0.21, 0.44, 0.33, 0.41, 0.24, 0.15, 0.48, 0.35])


Comparison of actual and predicted accident risk on training data (rounded to 2 decimal places):


,Actual Accident Risk,Predicted Accident Risk
0,0.13,0.13
1,0.35,0.32
2,0.30,0.42
3,0.21,0.13
4,0.56,0.43
5,0.61,0.63
6,0.20,0.37
7,0.16,0.16
8,0.18,0.19
9,0.15,0.21


In [23]:
# Load the original test data again to get the 'id' column
# Assuming the original test data with 'id' is still available from the initial loading step
try:
    original_test_df = pd.read_csv('test.csv')
except FileNotFoundError:
    print("Error: original test.csv not found. Please ensure it's in the correct directory.")
    original_test_df = None # Set to None if file not found

if original_test_df is not None:
    # Create a submission DataFrame
    # The submission file typically requires an 'id' column and the predicted target variable
    submission_df = pd.DataFrame({'id': original_test_df['id'], 'accident_risk': test_predictions})

    # Display the first few rows of the submission DataFrame
    print("Submission DataFrame head:")
    display(submission_df.head())
else:
    print("Could not create submission DataFrame as original test data was not loaded.")

Submission DataFrame head:


,id,accident_risk
0,517754,0.365056
1,517755,0.157263
2,517756,0.208640
3,517757,0.436072
4,517758,0.329073


In [24]:
# Save the submission DataFrame to a CSV file
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Submission file '{submission_filename}' created successfully.")

# Provide a link to download the file
from google.colab import files
files.download(submission_filename)

Submission file 'submission.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
print("Shape of the submission DataFrame:", submission_df.shape)

Shape of the submission DataFrame: (172585, 2)


# Task
Analyze the "https://www.kaggle.com/competitions/playground-series-s5e10" dataset by dropping the 'id' column, building a simple decision tree with 5-fold cross-validation and GridSearchCV, comparing prediction probabilities with 'accident_risk', trying a regression model to find accident risk, comparing predicted and actual values up to 2 decimal points, building a linear regression model on the whole data to make predictions on the test data, storing predictions in the sample submission format, converting predictions to a CSV file for download, and finally using PyCaret to compare different regression models to find the one with the lowest error.

## Install pycaret

### Subtask:
Install the PyCaret library.


**Reasoning**:
Install the full version of the PyCaret library using pip.



In [26]:
%pip install pycaret[full]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 53.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of flask to determine which version is compatible with other requirements. This could take a while.
     ━━━

## Setup pycaret environment

### Subtask:
Set up the PyCaret environment for the regression task.


**Reasoning**:
Set up the PyCaret environment for the regression task by importing the `setup` function and calling it with the training data and target variable.



In [1]:
from pycaret.regression import setup

# Set up the PyCaret environment
reg_setup = setup(data=train_df_encoded, target='accident_risk', session_id=123)

RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')

## Setup pycaret environment

### Subtask:
Setup pycaret environment for the regression task.

**Reasoning**:
Set up the PyCaret environment for the regression task by importing the `setup` function and calling it with the training data and target variable.

In [2]:
from pycaret.regression import setup

# Set up the PyCaret environment
reg_setup = setup(data=train_df_encoded, target='accident_risk', session_id=123)

RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')

## Setup pycaret environment

### Subtask:
Setup pycaret environment with a supported Python version for the regression task.


**Reasoning**:
Set up the PyCaret environment for the regression task with the correct Python version.



In [28]:
from pycaret.regression import setup

# Set up the PyCaret environment
reg_setup = setup(data=train_df_encoded, target='accident_risk', session_id=123)

RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')

## Setup pycaret environment

### Subtask:
Setup pycaret environment with a supported Python version for the regression task.


## Summary:

### Data Analysis Key Findings

*   The installation of the `pycaret[full]` library was successful, although some dependency conflicts were noted.
*   Attempts to set up the PyCaret environment for regression failed due to an incompatibility between the current Python version (3.12) and the supported versions by PyCaret (3.9, 3.10, 3.11).

### Insights or Next Steps

*   To proceed with the PyCaret analysis, the Python environment would need to be downgraded to a compatible version (3.9, 3.10, or 3.11).


# Task
Compare different regression models to find the one with the lowest error using cross-validation.

## Import necessary libraries

### Subtask:
Import the required regression models and evaluation metrics from scikit-learn.


**Reasoning**:
Import the required regression models, evaluation metrics, and cross-validation function from scikit-learn to prepare for model comparison.



In [3]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Define models

### Subtask:
Instantiate a few different regression models to compare (e.g., Linear Regression, Ridge, Lasso, Decision Tree Regressor, Random Forest Regressor).


**Reasoning**:
Instantiate different regression models and store them in a dictionary.



In [4]:
# Instantiate different regression models
ridge_model = Ridge(random_state=42)
lasso_model = Lasso(random_state=42)
dt_regressor = DecisionTreeRegressor(random_state=42)
rf_regressor = RandomForestRegressor(random_state=42)

# Store the models in a dictionary
regression_models = {
    'Ridge': ridge_model,
    'Lasso': lasso_model,
    'Decision Tree Regressor': dt_regressor,
    'Random Forest Regressor': rf_regressor
}

print("Regression models instantiated and stored in a dictionary:")
print(regression_models)

Regression models instantiated and stored in a dictionary:
{'Ridge': Ridge(random_state=42), 'Lasso': Lasso(random_state=42), 'Decision Tree Regressor': DecisionTreeRegressor(random_state=42), 'Random Forest Regressor': RandomForestRegressor(random_state=42)}


## Train and evaluate models

### Subtask:
Train each model on the training data and evaluate their performance using appropriate regression metrics (e.g., MAE, MSE, R2) using cross-validation.


**Reasoning**:
Iterate through the defined regression models, perform cross-validation for each using different scoring metrics, calculate the mean scores, and store them for comparison.



In [6]:
import pandas as pd

# Load the training and test data
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

except FileNotFoundError:
    print("Make sure the train.csv and test.csv files were extracted correctly.")

# Drop the 'id' column from both dataframes
train_df = train_df.drop('id', axis=1)
test_df = test_df.drop('id', axis=1)

# Identify categorical columns
categorical_cols = train_df.select_dtypes(include=['object', 'bool']).columns

# Apply one-hot encoding using pandas get_dummies
train_df_encoded = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Ensure columns match between train and test after encoding
# Add missing columns to the test set and fill with 0
missing_cols = set(train_df_encoded.columns) - set(test_df_encoded.columns)
for c in missing_cols:
    test_df_encoded[c] = 0
# Ensure the order of columns is the same
test_df_encoded = test_df_encoded[train_df_encoded.columns]

# Separate features (X) and target variable (y)
X_train = train_df_encoded.drop('accident_risk', axis=1)
y_train = train_df_encoded['accident_risk']
X_test = test_df_encoded.drop('accident_risk', axis=1) # Drop the target from test as well

from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Instantiate different regression models
ridge_model = Ridge(random_state=42)
lasso_model = Lasso(random_state=42)
dt_regressor = DecisionTreeRegressor(random_state=42)
rf_regressor = RandomForestRegressor(random_state=42)

# Store the models in a dictionary
regression_models = {
    'Ridge': ridge_model,
    'Lasso': lasso_model,
    'Decision Tree Regressor': dt_regressor,
    'Random Forest Regressor': rf_regressor
}

# Dictionary to store the mean cross-validation scores for each model
model_performance = {}

# Iterate through the regression models
for model_name, model in regression_models.items():
    print(f"Evaluating {model_name}...")

    # Perform 5-fold cross-validation
    # Using negative MAE, negative MSE, and R2 for scoring
    cv_mae_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    cv_mse_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_r2_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    # Calculate the mean of the cross-validation scores
    mean_mae = -cv_mae_scores.mean() # Negate to get positive MAE
    mean_mse = -cv_mse_scores.mean() # Negate to get positive MSE
    mean_r2 = cv_r2_scores.mean()

    # Store the mean scores
    model_performance[model_name] = {
        'Mean MAE': mean_mae,
        'Mean MSE': mean_mse,
        'Mean R2': mean_r2
    }

    print(f"{model_name} - Mean MAE: {mean_mae:.4f}, Mean MSE: {mean_mse:.4f}, Mean R2: {mean_r2:.4f}")

# Display the comparison of model performance
print("\nRegression Model Performance Comparison (Mean Cross-Validation Scores):")
performance_df = pd.DataFrame(model_performance).T # Transpose for better readability
display(performance_df)

Evaluating Ridge...
Ridge - Mean MAE: 0.0584, Mean MSE: 0.0054, Mean R2: 0.8050
Evaluating Lasso...
Lasso - Mean MAE: 0.1300, Mean MSE: 0.0266, Mean R2: 0.0408
Evaluating Decision Tree Regressor...
Decision Tree Regressor - Mean MAE: 0.0642, Mean MSE: 0.0069, Mean R2: 0.7508
Evaluating Random Forest Regressor...
Random Forest Regressor - Mean MAE: 0.0463, Mean MSE: 0.0035, Mean R2: 0.8720

Regression Model Performance Comparison (Mean Cross-Validation Scores):


,Mean MAE,Mean MSE,Mean R2
Ridge,0.058354,0.005402,0.804954
Lasso,0.130024,0.026564,0.040806
Decision Tree Regressor,0.064160,0.006901,0.750825
Random Forest Regressor,0.046285,0.003546,0.871960


## Compare models

### Subtask:
Display and compare the evaluation metrics for each model to identify the best performing one.

**Reasoning**:
Display the performance_df DataFrame to show the comparison of evaluation metrics for each model.

In [10]:
# Display the performance_df DataFrame
print("Regression Model Performance Comparison (Mean Cross-Validation Scores):")
display(performance_df)

Regression Model Performance Comparison (Mean Cross-Validation Scores):


,Mean MAE,Mean MSE,Mean R2
Ridge,0.058354,0.005402,0.804954
Lasso,0.130024,0.026564,0.040806
Decision Tree Regressor,0.064160,0.006901,0.750825
Random Forest Regressor,0.046285,0.003546,0.871960


## Create and Save Random Forest Submission File

### Subtask:
Format the Random Forest test predictions into a submission file and save it as a CSV.

**Reasoning**:
Load the original test data to get the 'id' column, create a submission DataFrame with the 'id' and Random Forest predictions, display the head, and save the DataFrame to a CSV file named 'rf_submission.csv' for download.

In [9]:
import pandas as pd
from google.colab import files

# Load the original test data again to get the 'id' column
# Assuming the original test data with 'id' is still available from the initial loading step
try:
    original_test_df = pd.read_csv('test.csv')
except FileNotFoundError:
    print("Error: original test.csv not found. Please ensure it's in the correct directory.")
    original_test_df = None # Set to None if file not found

if original_test_df is not None:
    # Create a submission DataFrame using Random Forest predictions
    # The submission file typically requires an 'id' column and the predicted target variable
    rf_submission_df = pd.DataFrame({'id': original_test_df['id'], 'accident_risk': rf_test_predictions})

    # Display the first few rows of the submission DataFrame
    print("Random Forest Submission DataFrame head:")
    display(rf_submission_df.head())

    # Save the submission DataFrame to a CSV file
    submission_filename = 'rf_submission.csv'
    rf_submission_df.to_csv(submission_filename, index=False)

    print(f"\nSubmission file '{submission_filename}' created successfully.")

    # Provide a link to download the file
    files.download(submission_filename)

else:
    print("\nCould not create submission DataFrame as original test data was not loaded.")

Random Forest Submission DataFrame head:


,id,accident_risk
0,517754,0.33555
1,517755,0.12690
2,517756,0.17075
3,517757,0.31910
4,517758,0.40160



Submission file 'rf_submission.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Train Random Forest Regressor and Make Predictions

### Subtask:
Train the Random Forest Regressor model and make predictions on the test data.

**Reasoning**:
Instantiate and train the Random Forest Regressor model on the entire training data, then use the trained model to make predictions on the test data.

In [8]:
from sklearn.ensemble import RandomForestRegressor

# Instantiate the Random Forest Regressor model (using default parameters for now)
rf_model = RandomForestRegressor(random_state=42)

# Train the model on the entire training data
rf_model.fit(X_train, y_train)

# Make predictions on the test data
rf_test_predictions = rf_model.predict(X_test)

print("Random Forest Regressor model trained and predictions made on test data.")
print("First 10 test predictions:", rf_test_predictions[:10].round(2))

Random Forest Regressor model trained and predictions made on test data.
First 10 test predictions: [0.34 0.13 0.17 0.32 0.4  0.42 0.26 0.19 0.37 0.32]


## Compare models

### Subtask:
Display and compare the evaluation metrics for each model to identify the best performing one.


**Reasoning**:
Display the performance_df DataFrame to show the comparison of evaluation metrics for each model.



In [7]:
# Display the performance_df DataFrame
print("Regression Model Performance Comparison (Mean Cross-Validation Scores):")
display(performance_df)

Regression Model Performance Comparison (Mean Cross-Validation Scores):


,Mean MAE,Mean MSE,Mean R2
Ridge,0.058354,0.005402,0.804954
Lasso,0.130024,0.026564,0.040806
Decision Tree Regressor,0.064160,0.006901,0.750825
Random Forest Regressor,0.046285,0.003546,0.871960


## Summary:

### Data Analysis Key Findings

*   The Random Forest Regressor model achieved the lowest Mean MAE (0.0463) and Mean MSE (0.0035) across the cross-validation folds.
*   The Random Forest Regressor also had the highest Mean R2 score (0.8720), indicating it explains the largest proportion of the variance in the target variable.
*   The Lasso model performed significantly worse than the other models, with much higher MAE and MSE and a very low R2 score.

### Insights or Next Steps

*   Based on the cross-validation results, the Random Forest Regressor is the best-performing model for this dataset and task.
*   The next step could involve hyperparameter tuning for the Random Forest Regressor to potentially further improve its performance.
